In [4]:
import os 

keyPath = os.path.join('src', "key.properties")
if os.path.exists(keyPath):
    print("key exists")

else:
    print("key.properties file not found. Please create the file and add your key.")
    

key exists


In [5]:
import importlib
import src.mylib
importlib.reload(src.mylib)   # mylib.py를 수정했을 때 커널 재시작 없이 다시 불러오기

keyPath = os.path.join(os.getcwd(), 'src', "key.properties")
key = src.mylib.get_Key(keyPath)
print("키 불러옴:", key[:6] + "***")   # 키 전체는 출력하지 않기


키 불러옴: RGAPI-***


## 1. Riot ID로 puuid 조회 (Account-V1)
소환사 이름으로 조회하는 `by-name` API는 폐지되었다. 이제 **Riot ID(이름#태그)** 로 puuid를 먼저 얻는다.
- host: `asia` (대륙)
- path: `/riot/account/v1/accounts/by-riot-id/{이름}/{태그}`


In [6]:
game_name = "붉은기름콩"
tag_line = "5083"   # '#' 뒤의 태그로 바꾸기 (게임 클라이언트 프로필에서 확인)

account = src.mylib.get_account(key, game_name, tag_line)
puuid = account["puuid"]
account


{'puuid': '0y72ZLsOiQ8YyVFMJXk0PhYQi57cVgUi1QLFUHF6NZGjNdZ5EX4eC_SQjB2p8NcH9Pn7BDNE3nrOOA',
 'gameName': '붉은기름콩',
 'tagLine': '5083'}

## 2. puuid로 소환사 정보 조회 (Summoner-V4)
- host: `kr` (서버)
- path: `/lol/summoner/v4/summoners/by-puuid/{puuid}`


In [7]:
summoner = src.mylib.get_summoner(key, puuid)
summoner


{'puuid': '0y72ZLsOiQ8YyVFMJXk0PhYQi57cVgUi1QLFUHF6NZGjNdZ5EX4eC_SQjB2p8NcH9Pn7BDNE3nrOOA',
 'profileIconId': 6,
 'revisionDate': 1795529728000,
 'summonerLevel': 120}

## 3. 최근 경기 목록 조회 (Match-V5)
- host: `asia` (대륙)
- path: `/lol/match/v5/matches/by-puuid/{puuid}/ids`


In [28]:
match_ids = src.mylib.get_match_ids(key, puuid, count=40,queue=420)   # queue=420 을 주면 솔로랭크만
match_ids


['KR_8471970706',
 'KR_8471916345',
 'KR_8471898316',
 'KR_8471840406',
 'KR_8471763950',
 'KR_8471712877',
 'KR_8471688716',
 'KR_8470934205',
 'KR_8467624724',
 'KR_8467542964',
 'KR_8465726988',
 'KR_8465677109',
 'KR_8465634979',
 'KR_8465318568',
 'KR_8465298858',
 'KR_8465292632',
 'KR_8465284849',
 'KR_8465281588',
 'KR_8465276040',
 'KR_8463553678',
 'KR_8463276523',
 'KR_8456876425',
 'KR_8455917413',
 'KR_8455905259',
 'KR_8422791281',
 'KR_8421473861',
 'KR_8421407962',
 'KR_8421369751',
 'KR_8419356654',
 'KR_8419290770',
 'KR_8419236434',
 'KR_8419199683',
 'KR_8412373898',
 'KR_8412316871',
 'KR_8412292676',
 'KR_8412246840',
 'KR_8412224688',
 'KR_8412188969',
 'KR_8411457421',
 'KR_8411369607']

In [47]:
match_history = []
for i in range(40):
    m = src.mylib.get_match(key, match_ids[i])
    match_history.append(m)

match = match_history[9]   # 첫 번째 경기만 확인

In [48]:
match["info"]["participants"][0]["championName"], match["info"]["participants"][0]["win"]

('Urgot', False)

In [49]:
match["info"]["participants"][5]["championName"], match["info"]["participants"][5]["win"]


('Olaf', True)

In [50]:
import pandas as pd

df = pd.DataFrame(match["info"]["participants"])
df[["riotIdGameName", "championName", "kills", "deaths", "assists", "win"]]

,riotIdGameName,championName,kills,deaths,assists,win
0,천연기념물 쉔,Urgot,4,12,4,False
1,진시황,Pantheon,12,8,6,False
2,김덕배,Anivia,3,4,9,False
3,뭐꼬이거,Syndra,14,5,13,False
4,빙준이다,Swain,4,8,18,False
5,붉은기름콩,Olaf,11,8,3,True
6,현 라,Amumu,8,12,15,True
7,정글서폿잘해야이기는게임,Azir,7,2,6,True
8,가영e,Ezreal,7,9,12,True
9,유토체스,Senna,4,8,21,True


In [51]:
match.keys()

dict_keys(['metadata', 'info'])

In [52]:
match["info"].keys()

dict_keys(['endOfGameResult', 'gameCreation', 'gameDuration', 'gameEndTimestamp', 'gameId', 'gameMode', 'gameName', 'gameStartTimestamp', 'gameType', 'gameVersion', 'mapId', 'participants', 'platformId', 'queueId', 'teams', 'tournamentCode'])

In [53]:
import json
with open("match_sample.json", "w", encoding="utf-8") as f:
    json.dump(match, f, ensure_ascii=False, indent=2)


In [54]:
flat = pd.json_normalize(match["info"]["participants"])
flat.columns.tolist()   # 'challenges.kda', 'perks.styles' 같은 이름으로 펼쳐짐


['PlayerScore0',
 'PlayerScore1',
 'PlayerScore10',
 'PlayerScore11',
 'PlayerScore2',
 'PlayerScore3',
 'PlayerScore4',
 'PlayerScore5',
 'PlayerScore6',
 'PlayerScore7',
 'PlayerScore8',
 'PlayerScore9',
 'allInPings',
 'assistMePings',
 'assists',
 'baronKills',
 'basicPings',
 'causedGameEndFromIGNBSurrender',
 'champExperience',
 'champLevel',
 'championId',
 'championName',
 'championTransform',
 'commandPings',
 'consumablesPurchased',
 'damageDealtToBuildings',
 'damageDealtToEpicMonsters',
 'damageDealtToObjectives',
 'damageDealtToTurrets',
 'damageSelfMitigated',
 'dangerPings',
 'deaths',
 'detectorWardsPlaced',
 'doubleKills',
 'dragonKills',
 'eligibleForProgression',
 'enemyMissingPings',
 'enemyVisionPings',
 'firstBloodAssist',
 'firstBloodKill',
 'firstTowerAssist',
 'firstTowerKill',
 'gameEndedInEarlySurrender',
 'gameEndedInIGNBSurrender',
 'gameEndedInSurrender',
 'getBackPings',
 'goldEarned',
 'goldSpent',
 'holdPings',
 'individualPosition',
 'inhibitorKills',


In [55]:
_start_index = 0
_end_index = 20

data = src.mylib.get_match_ids(key, puuid,
                               start=_start_index,
                               count=_end_index - _start_index)
print(data)



['KR_8471970706', 'KR_8471916345', 'KR_8471898316', 'KR_8471840406', 'KR_8471763950', 'KR_8471712877', 'KR_8471688716', 'KR_8470934205', 'KR_8467624724', 'KR_8467542964', 'KR_8465726988', 'KR_8465677109', 'KR_8465634979', 'KR_8465318568', 'KR_8465298858', 'KR_8465292632', 'KR_8465284849', 'KR_8465281588', 'KR_8465276040', 'KR_8463647177']


In [56]:
data = src.mylib.riot_get(key, "asia",
                          f"/lol/match/v5/matches/by-puuid/{puuid}/ids",
                          {"start": 0, "count": 20})
print(data)


['KR_8471970706', 'KR_8471916345', 'KR_8471898316', 'KR_8471840406', 'KR_8471763950', 'KR_8471712877', 'KR_8471688716', 'KR_8470934205', 'KR_8467624724', 'KR_8467542964', 'KR_8465726988', 'KR_8465677109', 'KR_8465634979', 'KR_8465318568', 'KR_8465298858', 'KR_8465292632', 'KR_8465284849', 'KR_8465281588', 'KR_8465276040', 'KR_8463647177']


In [57]:
import json
a = json.load(open("match_sample.json", "r", encoding="utf-8"))

In [58]:
b = a["info"]["participants"][5]
c = a["info"]["participants"][0]

In [59]:
b.keys()


dict_keys(['PlayerBehavior', 'PlayerScore0', 'PlayerScore1', 'PlayerScore10', 'PlayerScore11', 'PlayerScore2', 'PlayerScore3', 'PlayerScore4', 'PlayerScore5', 'PlayerScore6', 'PlayerScore7', 'PlayerScore8', 'PlayerScore9', 'allInPings', 'assistMePings', 'assists', 'baronKills', 'basicPings', 'causedGameEndFromIGNBSurrender', 'challenges', 'champExperience', 'champLevel', 'championId', 'championName', 'championTransform', 'commandPings', 'consumablesPurchased', 'damageDealtToBuildings', 'damageDealtToEpicMonsters', 'damageDealtToObjectives', 'damageDealtToTurrets', 'damageSelfMitigated', 'dangerPings', 'deaths', 'detectorWardsPlaced', 'doubleKills', 'dragonKills', 'eligibleForProgression', 'enemyMissingPings', 'enemyVisionPings', 'firstBloodAssist', 'firstBloodKill', 'firstTowerAssist', 'firstTowerKill', 'gameEndedInEarlySurrender', 'gameEndedInIGNBSurrender', 'gameEndedInSurrender', 'getBackPings', 'goldEarned', 'goldSpent', 'holdPings', 'individualPosition', 'inhibitorKills', 'inhibit

In [60]:
mygold =int(b.get('goldEarned'))
enemygold =int(c.get('goldEarned'))
print(mygold - enemygold)

4249


In [77]:
def gold_diff(match, puuid):
    """한 경기에서 나와 맞라이너의 골드 차이를 반환. 비교할 수 없으면 None."""
    participants = match["info"]["participants"]

    # 1) 나 찾기
    me = next(p for p in participants if p["puuid"] == puuid)

    # 2) 상대 팀에서 같은 포지션인 사람 찾기
    enemy = next((p for p in participants
                  if p["teamId"] != me["teamId"]
                  and p["teamPosition"] == me["teamPosition"]), None)
    if enemy is None or me["teamPosition"] == "":   # 칼바람 등 포지션 없는 경기
        return None                                  # continue 대신 return

    return me["goldEarned"] - enemy["goldEarned"]   # 쉼표 없이

    

In [78]:
import pandas as pd

rows = []
for match in match_history:
    participants = match["info"]["participants"]

    # 1) 나 찾기
    
    me = next(p for p in participants if p["puuid"] == puuid)

    # 2) 상대 팀에서 같은 포지션인 사람 찾기
    enemy = next((p for p in participants
                  if p["teamId"] != me["teamId"]
                  and p["teamPosition"] == me["teamPosition"]), None)
    if enemy is None or me["teamPosition"] == "":   # 칼바람 등 포지션 없는 경기는 건너뜀
        continue

    rows.append({
        "match_id":   match["metadata"]["matchId"],
        "position":   me["teamPosition"],
        "my_champ":   me["championName"],
        "enemy_champ": enemy["championName"],
        "my_gold":    me["goldEarned"],
        "enemy_gold": enemy["goldEarned"],
        "gold_diff":  me["goldEarned"] - enemy["goldEarned"],
        "win":        me["win"],
    })

gold_df = pd.DataFrame(rows)
gold_df


,match_id,position,my_champ,enemy_champ,my_gold,enemy_gold,gold_diff,win
0,KR_8471970706,TOP,DrMundo,Veigar,15791,15660,131,True
1,KR_8471916345,TOP,Gnar,Renekton,10009,14000,-3991,False
2,KR_8471898316,TOP,Ornn,Tryndamere,503,923,-420,False
3,KR_8471840406,TOP,Olaf,Vayne,9997,12811,-2814,True
4,KR_8471763950,TOP,Zaahen,Yone,13108,10483,2625,True
5,KR_8471712877,TOP,Jayce,Sylas,12553,14461,-1908,True
6,KR_8471688716,TOP,Olaf,Jax,5488,5227,261,True
7,KR_8470934205,TOP,Olaf,Camille,17039,13977,3062,False
8,KR_8467624724,JUNGLE,Diana,Jayce,13180,16031,-2851,True
9,KR_8467542964,TOP,Olaf,Urgot,16979,12730,4249,True


In [79]:
import pandas as pd

rows = []
for match in match_history:
    # 5분 미만 경기(다시하기)는 건너뜀
    if match["info"]["gameDuration"] < 300:
        continue

    participants = match["info"]["participants"]

    # 나 찾기
    me = next(p for p in participants if p["puuid"] == puuid)

    # 팀별 골드 합계
    my_team_gold    = sum(p["goldEarned"] for p in participants if p["teamId"] == me["teamId"])
    enemy_team_gold = sum(p["goldEarned"] for p in participants if p["teamId"] != me["teamId"])

    # 경기당 1줄 저장
    rows.append({
        "match_id":          match["metadata"]["matchId"],
        "my_team_gold":      my_team_gold,
        "enemy_team_gold":   enemy_team_gold,
        "diff_of_team_gold": my_team_gold - enemy_team_gold,
        "win":               me["win"],
    })

team_gold_df = pd.DataFrame(rows)
print(team_gold_df)


         match_id  my_team_gold  enemy_team_gold  diff_of_team_gold    win
0   KR_8471970706         86450            84337               2113   True
1   KR_8471916345         51860            67722             -15862  False
2   KR_8471840406         60254            50770               9484   True
3   KR_8471763950         68305            57187              11118   True
4   KR_8471712877         60632            53439               7193   True
5   KR_8471688716         30120            25881               4239   True
6   KR_8470934205         78730            77042               1688  False
7   KR_8467624724         69182            65770               3412   True
8   KR_8467542964         80620            74154               6466   True
9   KR_8465726988         73710            76073              -2363  False
10  KR_8465677109        104386            94865               9521   True
11  KR_8465634979         33643            28051               5592   True
12  KR_8465318568        

In [80]:
len(match_history)

40

In [81]:
for row in rows:      # 한 경기씩 꺼내기
    if row["diff_of_team_gold"] > 0 and row["win"]== False:
        print(f"match_id: {row['match_id']}, diff_of_team_gold: {row['diff_of_team_gold']}, win: {row['win']}")

match_id: KR_8470934205, diff_of_team_gold: 1688, win: False
match_id: KR_8455917413, diff_of_team_gold: 875, win: False


In [82]:
lost_with_lead = team_gold_df[(team_gold_df["diff_of_team_gold"] > 0) & (~team_gold_df["win"])]
print(lost_with_lead)


         match_id  my_team_gold  enemy_team_gold  diff_of_team_gold    win
6   KR_8470934205         78730            77042               1688  False
21  KR_8455917413         83038            82163                875  False


In [86]:
gold_diff(match_history[22], puuid)

7358

In [85]:
gold_diff(match_history[7], puuid)

3062